# Vektorisierung der Review-Texte

In diesem Notebook werden die vorverarbeiteten Reviews mit Bag-of-Words und TF-IDF in numerische Darstellungen umgewandelt.

In [1]:
import pandas as pd

file_path = "../data/processed/processed_reviews.jsonlines"

df_model = pd.read_json(
    file_path,
    lines=True
)

df_model.shape

(10988, 5)

In [2]:
df_model.columns.tolist()

['rating', 'review', 'tokens', 'token_count', 'clean_text']

## Bag-of-Words

Mit Bag-of-Words werden die bereinigten Review-Texte anhand der Häufigkeit ihrer Wörter in eine numerische Matrix umgewandelt.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.95,
    max_features=5000
)

bow_matrix = bow_vectorizer.fit_transform(
    df_model["clean_text"]
)

bow_matrix.shape

(10988, 4528)

In [4]:
bow_terms = bow_vectorizer.get_feature_names_out()

bow_terms[:30]

array(['aa', 'aaa', 'ability', 'abit', 'able', 'absence', 'absolute',
       'absolutely', 'absolutley', 'absolutly', 'absurd', 'abuse',
       'abusing', 'abysmal', 'acceleration', 'accept', 'acceptable',
       'accepted', 'access', 'accident', 'accidentally', 'accidently',
       'accompany', 'accomplish', 'according', 'account', 'accuracy',
       'accurate', 'accurately', 'achieve'], dtype=object)

In [5]:
first_review_counts = bow_matrix[0].toarray().flatten()

first_review_bow = pd.DataFrame({
    "word": bow_terms,
    "count": first_review_counts
})

first_review_bow = first_review_bow[
    first_review_bow["count"] > 0
].sort_values("count", ascending=False)

first_review_bow

,word,count
1780,hacker,2
871,crashed,1
392,besides,1
1046,die,1
1296,eventhough,1
1601,fun,1
958,death,1
1619,game,1
1704,got,1
1781,hacking,1


## TF-IDF

TF-IDF gewichtet Wörter anhand ihrer Häufigkeit in einem einzelnen Review und ihrer Seltenheit im gesamten Datensatz. Wörter, die in einem Review häufig, im Gesamtdatensatz jedoch selten vorkommen, erhalten ein höheres Gewicht.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.95,
    max_features=5000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df_model["clean_text"]
)

tfidf_matrix.shape

(10988, 4528)

In [7]:
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

first_review_tfidf_values = tfidf_matrix[0].toarray().flatten()

first_review_tfidf = pd.DataFrame({
    "word": tfidf_terms,
    "tfidf": first_review_tfidf_values
})

first_review_tfidf = first_review_tfidf[
    first_review_tfidf["tfidf"] > 0
].sort_values("tfidf", ascending=False)

first_review_tfidf.head(15)

,word,tfidf
1296,eventhough,0.283482
1780,hacker,0.265690
2102,invalid,0.257735
2337,loop,0.257735
3732,solved,0.241765
2049,input,0.232185
3114,putting,0.208513
958,death,0.198934
1781,hacking,0.198498
392,besides,0.195207


## Vergleich von Bag-of-Words und TF-IDF

Die Bag-of-Words-Darstellung speichert die absolute Häufigkeit der Wörter in jedem Review. Dadurch werden häufig verwendete Begriffe unabhängig von ihrer Verbreitung im gesamten Datensatz stark gewichtet.

TF-IDF berücksichtigt zusätzlich die Dokumenthäufigkeit. Wörter, die in vielen Reviews vorkommen, erhalten ein geringeres Gewicht. Spezifische Begriffe können dadurch stärker hervortreten. Gleichzeitig können seltene Schreibfehler vergleichsweise hohe Gewichte erhalten.